In [ ]:
import pandas as pd
import yaml
try:
    from yaml import Cloader as Loader
except ImportError:

    from yaml import Loader
from pyDTDM import *
import gplately


import pyproj
import os

# Point pyproj to the correct PROJ data directory
os.environ["PROJ_LIB"] = "<CONDA>/envs/EBMTest311/share/proj"
pyproj.datadir.set_data_dir(os.environ["PROJ_LIB"])

# crs = CRS.from_epsg(4326)
# print(crs)


In [ ]:
grid_data=pd.read_csv("<DATA_ROOT>/CopperLithium/NWMexico/PUBaggingModel/grid_data_fz_seamount_LIP.csv")

In [ ]:
grid_data.columns

In [ ]:
config_file="inputfile.yaml"
with open(config_file) as f:
    PARAMS = yaml.load(f, Loader=Loader)
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print(" Parameters set from %s" % config_file)
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")


In [ ]:
# Input Files 
MODEL_NAME=PARAMS['InputFiles']['plate_kinematics']['model_name'] # model name
MODEL_DIR = PARAMS['InputFiles']['plate_kinematics']['model_dir']  ## plate model location
topology_filenames =[f"{MODEL_DIR}/{i}" for i in PARAMS['InputFiles']['plate_kinematics']['topology_files']]
rotation_filenames = [f"{MODEL_DIR}/{i}" for i in PARAMS['InputFiles']['plate_kinematics']['rotation_files']]


coastlines = f"{PARAMS['InputFiles']['plate_kinematics']['coastline_file']}"
static_polygon_file=f"{PARAMS['InputFiles']['plate_kinematics']['static_polygon']}"
static_polygons = pygplates.FeatureCollection(static_polygon_file)
continents=f"{PARAMS['InputFiles']['plate_kinematics']['continents']}"
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print("Reading input file..... \n")
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print(f"Plate Model: {MODEL_NAME} \n")
print(f"Model Directory: {MODEL_DIR} \n")
print(f"Coastlines: {coastlines} \n")
print(f"Continents: {continents} \n")
print(f"Static Polygons: {static_polygon_file} \n")
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– \n")

In [ ]:
Paleomag_ID=PARAMS['Parameters']['paleomag_id']
Mantle_ID=PARAMS['Parameters']['mantle_optimised_id']

#The initial positions of crustal points are evenly distributed within the designated region. 
# At mesh refinement level zero, the points are approximately 20 degrees apart.
# Each increase in the density level results in a halving of the spacing between points.
MESH_REFINEMENT_LEVEL=PARAMS['Parameters']['mesh_refinement_level']  # higher refinement level will take longer time to run for optimisation 
WINDOW_SIZE=int(PARAMS['Parameters']['time_window_size'])
Weighted=PARAMS['Parameters']['weighted_mean']


NETCDF_GRID_RESOLUTION=PARAMS['GridParameters']['grid_spacing']  # in degree
ZLIB=PARAMS['GridParameters']['compression']['zlib'] 
COMPLEVEL=PARAMS['GridParameters']['compression']['complevel'] 

FROM_TIME=int(PARAMS['TimeParameters']['time_max'])
TO_TIME=int(PARAMS['TimeParameters']['time_min'])
TIME_STEPS=int(PARAMS['TimeParameters']['time_step'])




parallel=PARAMS['Parameters']['number_of_cpus']### No of core to use or None for single core


print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print("The following parameters are set-")
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print(f"Mantle Optmised Reference Frame ID: {Mantle_ID}")
print(f"Paleomagnetic Reference Frame ID: {Paleomag_ID} \n")

print(f"Moving Window Size: {WINDOW_SIZE}")
print(f"Weighted Mean: {Weighted}")

print(f"Mesh Refinement Level: {MESH_REFINEMENT_LEVEL}")
print(f"NetCDF GRID Resolution: {NETCDF_GRID_RESOLUTION}")
print(f"NetCDF Compression Level: {COMPLEVEL} \n")
print(f"Model Start Time: {FROM_TIME}")
print(f"Model End Time: {TO_TIME}")
print(f"Model Time Step: {TIME_STEPS}\n")


print(f"Number of CPU: {parallel}") # -1 means all the freely available CPU


print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")

In [ ]:
from plate_model_manager import PlateModelManager
pm_manager = PlateModelManager()
muller2019_model = pm_manager.get_model("Clennett2020_M2019", data_dir="plate-model-repo")
rotation_filenames = muller2019_model.get_rotation_model()
topology_filenames = muller2019_model.get_topologies()
static_polygons = muller2019_model.get_static_polygons()

In [ ]:

PK=PlateKinematicsParameters(topology_filenames, ## all the plate topologies files (as list of str)
                             rotation_filenames, ### all the rotation files (as list of str)
                             static_polygons,  ### as str
                             coastlines=coastlines, ## str
                             continents=continents, ## st
                             anchor_plate_id=Mantle_ID) ## int



In [ ]:
probability_grids="<DATA_ROOT>/CopperLithium/NorthAmerica/output/probability_grids"

In [ ]:
reconstruction_time=150

point_nc=xr.open_dataarray(f"{probability_grids}/grids_{reconstruction_time}.nc")
point_nc.name="probabilites"
point_df=point_nc.to_dataframe().dropna().reset_index()

In [ ]:
plt.scatter(point_df['Longitude'], point_df['Latitude'],c=point_df['probabilites'])

In [ ]:
def present_day_probs(lons,lats,reconstruction_time):
        
        lat=np.array(lats).astype(float)
        lon=np.array(lons).astype(float)
        initial_points=pygplates.MultiPointOnSphere(zip(lat,lon))
        # Elevation=np.ones(len(lat))
        initial_time = reconstruction_time
        # time_increment = timesteps
        PK.topological_model = pygplates.TopologicalModel(PK.topology_features, PK.rotation_model,anchor_plate_id=PK.anchor_plate_id)
        # no_deactivate_points = NoDeactivatePoints()

        # deactivate_points = (
        #     pygplates.ReconstructedGeometryTimeSpan
        # ).DefaultDeactivatePoints(
        #     deactivate_points_that_fall_outside_a_network=False
        # )
        
        # reconstructed_time_span = PK.topological_model.reconstruct_geometry(initial_points, initial_time, 
        #                                                                  time_increment=1,
        #                                                                  deactivate_points=deactivate_points )

        reconstructed_time_span = PK.topological_model.reconstruct_geometry(initial_points, initial_time, 
                                                                         time_increment=1,
                                                                         deactivate_points=None) 
        
        reconstructed_points = reconstructed_time_span.get_geometry_points(0,return_inactive_points=True)

        lats=[]
        lons=[]
        indices=[]
        i=0
        for rp in reconstructed_points:
            if rp !=None:
                indices.append(i)
                lats.append(rp.to_lat_lon()[0])
                lons.append(rp.to_lat_lon()[1])
            i+=1
        return lons,lats,indices


       



In [ ]:
DEFAULT_CRS="EPSG:4326"

combined_probs=[]
for reconstruction_time in range(FROM_TIME,TO_TIME,-TIME_STEPS):
# for reconstruction_time in range(160,170,TIME_STEPS):
    print(f"Working on {reconstruction_time}")
    # point_nc=xr.open_dataarray(f"{probability_grids}/probability_grid_{reconstruction_time}Ma.nc")
    # point_df=point_nc.to_dataframe().dropna().reset_index()

    point_nc=xr.open_dataarray(f"{probability_grids}/grids_{reconstruction_time}.nc")
    point_nc.name="probabilites"
    point_df=point_nc.to_dataframe().dropna().reset_index()
    rlons,rlats,indices=present_day_probs(point_df['Longitude'].values,point_df['Latitude'].values,reconstruction_time=reconstruction_time)
   
    df=pd.DataFrame()
    df['Present Latitude']=rlats
    df['Present Longitude']=rlons
    df['Latitude']=point_df['Latitude']
    df['Longitude']=point_df['Longitude']
    df['Age']=reconstruction_time
    df['probalities']=point_df["probabilites"]
    df['index']=indices
    # df['PlateID']=plate_id['PLATEID1']
    combined_probs.append(df)
    # break



In [ ]:
# DEFAULT_CRS="EPSG:4326"

# combined_probs=[]
# for reconstruction_time in range(TO_TIME,FROM_TIME,TIME_STEPS):
# # for reconstruction_time in range(160,170,TIME_STEPS):
#     print(f"Working on {reconstruction_time}")
#     point_nc=xr.open_dataarray(f"{probability_grids}/probability_grid_{reconstruction_time}Ma.nc")
#     point_df=point_nc.to_dataframe().dropna().reset_index()
#     gpts=gplately.Points(PK.model, point_df['lon'],point_df['lat'],age=reconstruction_time)
#     resolved_topologies = ptt.resolve_topologies.resolve_topologies_into_features(
#             PK.rotation_model,PK.topology_features, reconstruction_time)#,anchor_plate_id=PK.anchor_plate_id)
#     topologies, ridge_transforms, ridges, transforms, trenches, trench_left, trench_right, other = resolved_topologies

#     topologies_gdf=create_geodataframe_topologies(topologies, reconstruction_time)
#     topologies_gdf=topologies_gdf.set_crs(DEFAULT_CRS)

#     training_gdf=gpd.GeoDataFrame(point_df, geometry=gpd.points_from_xy(point_df['lon'],point_df['lat']))
#     training_gdf=training_gdf.set_crs(DEFAULT_CRS)

#     plate_id = gpd.sjoin_nearest(training_gdf, topologies_gdf, how='left')

#     # Ensure same length by removing duplicates
#     if len(plate_id) != len(training_gdf):
#         plate_id = plate_id.reset_index().drop_duplicates(subset='index', keep='first').set_index('index')

#     gpts = gplately.Points(PK.model, point_df['lon'], point_df['lat'],reconstruction_time,plate_id=plate_id['PLATEID1'].values,age=reconstruction_time)
#     # gpts = gplately.Points(PK.model, point_df['lon'], point_df['lat'],reconstruction_time,age=reconstruction_time)
#     rlons,rlats=gpts.reconstruct(time=0,return_array=True)
#     df=pd.DataFrame()
#     df['Present Latitude']=rlats
#     df['Present Longitude']=rlons
#     df['Latitude']=point_df['lat']
#     df['Longitude']=point_df['lon']
#     df['Age']=reconstruction_time
#     df['probalities']=point_df['z']
#     df['PlateID']=plate_id['PLATEID1']
#     combined_probs.append(df)



In [ ]:
combined_probs_df_save=pd.concat(combined_probs)
combined_probs_df_save.to_csv("<DATA_ROOT>/CopperLithium/NorthAmerica/output/combined_probability_grids.csv")

In [ ]:
combined_probs_df=pd.concat(combined_probs)


In [ ]:
grid_resolution=0.25
# Compute statistics on the grid
probs_counts = df_to_NetCDF(
    combined_probs_df['Present Longitude'],
    combined_probs_df['Present Latitude'],
    combined_probs_df['probalities'],
    statistic='count',
    grid_resolution=grid_resolution
)
probs_counts.name = "counts"

probs_mean = df_to_NetCDF(
    combined_probs_df['Present Longitude'],
    combined_probs_df['Present Latitude'],
    combined_probs_df['probalities'],
    statistic='mean',
    grid_resolution=grid_resolution
)
probs_mean.name = "mean"

probs_max = df_to_NetCDF(
    combined_probs_df['Present Longitude'],
    combined_probs_df['Present Latitude'],
    combined_probs_df['probalities'],
    statistic='max',
    grid_resolution=grid_resolution
)
probs_max.name = "max"

probs_median = df_to_NetCDF(
    combined_probs_df['Present Longitude'],
    combined_probs_df['Present Latitude'],
    combined_probs_df['probalities'],
    statistic='median',
    grid_resolution=grid_resolution
)
probs_median.name = "median"

probs_std = df_to_NetCDF(
    combined_probs_df['Present Longitude'],
    combined_probs_df['Present Latitude'],
    combined_probs_df['probalities'],
    statistic='std',
    grid_resolution=grid_resolution
)
probs_std.name = "std"



# probs_ero = df_to_NetCDF(
#     combined_probs_df['Present Longitude'],
#     combined_probs_df['Present Latitude'],
#     combined_probs_df['Erosion (m)'],
#     statistic='mean',
#     grid_resolution=0.5
# )
# probs_ero.name = "Erosion (m)"

# Combine all statistics into a single DataFrame
counts_df = probs_counts.to_dataframe().reset_index()
counts_df['mean'] = probs_mean.to_dataframe().reset_index()['mean']
counts_df['max'] = probs_max.to_dataframe().reset_index()['max']
counts_df['median'] = probs_median.to_dataframe().reset_index()['median']
counts_df['std'] = probs_std.to_dataframe().reset_index()['std']
# counts_df['erosion'] = probs_ero.to_dataframe().reset_index()['std']
# counts_df now contains: longitude, latitude, counts, mean, max, median, std
counts_df=counts_df[counts_df['counts']>=10]

In [ ]:
counts_gdf=gpd.GeoDataFrame(counts_df,geometry=gpd.points_from_xy(counts_df['Longitude'],counts_df['Latitude']))
counts_gdf=counts_gdf.set_crs(DEFAULT_CRS)
counts_gdf.to_file("<DATA_ROOT>/CopperLithium/NorthAmerica/output/combined_probability_stats.gpkg")

In [ ]:
probs_mean.name = "counts"
combined_probs_df = probs_mean.to_dataframe().reset_index()

In [ ]:
combined_probs_df=combined_probs_df[combined_probs_df['counts']>=10]

In [ ]:
plt.hist(flatten_list(probs_mean.values),bins=500)
plt.xlim([0,100])
plt.ylim([0,5000])

In [ ]:
probs_mean=df_to_NetCDF(combined_probs_df['Longitude'],combined_probs_df['Latitude'],combined_probs_df['probalities'],statistic='max',grid_resolution=0.5)
probs_mean.to_netcdf("<DATA_ROOT>/CopperLithium/NWMexico/combined_probability_max.nc")
probs_mean.plot(vmin=0)

In [ ]:
probs_mean=df_to_NetCDF(combined_probs_df['Present Longitude'],combined_probs_df['Present Latitude'],combined_probs_df['probalities'],statistic='mean',grid_resolution=0.5)
probs_mean.to_netcdf("<DATA_ROOT>/CopperLithium/NWMexico/combined_probability_mean.nc")

In [ ]:
probs_mean=df_to_NetCDF(combined_probs_df['Present Longitude'],combined_probs_df['Present Latitude'],combined_probs_df['probalities'],statistic='std',grid_resolution=0.5)
probs_mean.to_netcdf("<DATA_ROOT>/CopperLithium/NWMexico/combined_probability_std.nc")

In [ ]:
probs_mean.plot(vmin=0)

In [ ]:
plt.scatter(point_df['lon'],point_df['lat'])

In [ ]:
plt.scatter(rlons,rlats,c=point_df['z'])

In [ ]:
copper=gpd.read_file("<DATA_ROOT>/CopperLithium/NWMexico/Data/pcu_deps_pros.csv")

In [ ]:
len(copper['tonnage_mt'])

In [ ]:
copper['tonnage_mt']